# 订单拣选问题

**类别：** 路径规划

来源：[https://www.hexaly.com/templates/order-picking-problem](https://www.hexaly.com/templates/order-picking-problem)


## 问题描述

**订单拣选问题** 描述如下：需要拣选一组订单。拣货员从初始位置出发，拣取订单后返回初始位置卸货。我们考虑一个普通的矩形仓库，其中有一个用于卸货的单一仓库起点。该仓库起点也作为拣货员的初始位置。需要拣取的订单位于垂直通道的两侧，两侧均可到达。垂直通道由水平横向通道环绕，拣货员可通过水平横向通道在仓库内移动。拣货员可以纵向和横向移动。任意两个订单之间的距离使用曼哈顿距离计算。问题的目标是找到使完成订单所需距离（或时间）最小的拣选顺序。

	

### 建模要点

- 添加一个 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模拣选顺序
- 定义一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算行驶距离
- 获取 [list 变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 的值


## 数据

我们提供的订单拣选问题实例来自 [Theys et al. 基准](https://repository.uantwerpen.be/desktop/irua)。数据文件的格式如下：

- 第一行：订单数量
- 接下来若干行：任意两个订单（包括索引 0 处的初始位置）之间的距离矩阵。


## 模型

在订单拣选问题的 Hexaly 模型中，我们使用一个 list 决策变量表示拣选顺序。我们也将初始点视为一个待拣选订单。对 list 大小的约束确保所有订单都被拣取。

目标是最小化拣取所有订单所需的距离。为了计算该距离，我们使用一个 lambda 函数返回从一个订单到下一个订单的距离。我们通过对该 lambda 函数在所有订单位置上的求和来计算目标值。

由于拣货员的行驶路径是一个回路，因此不需要将 list 中的第一个元素约束为初始点。这可以在求解后的后处理阶段完成。为此，我们在模型声明中使用 **indexOf** 算子来恢复 list 中元素 0 的位置。然后我们可以将拣选顺序保存到文件中，从初始点开始。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

def read_elem(filename) :
    with open(filename) as f :
        return [str(elem) for elem in f.read().split()]

def read_instance(filename) :
    file_iterator = iter(read_elem(filename))
    nb_orders = int(next(file_iterator)) + 1
    distances_data = [None] * nb_orders
    for i in range(nb_orders) :
        distances_data[i] = [None] * nb_orders
        for j in range(nb_orders) :
            distances_data[i][j] = int(next(file_iterator))
    return nb_orders, distances_data

def main(input_file, output_file, time_limit) :
    # Read the instance from input_file
    nb_orders, distances_data = read_instance(input_file)
    
    with hexaly.optimizer.HexalyOptimizer() as optimizer :
        # Declare the model
        model = optimizer.model

        # Declare the list containing the picking order
        picking_list = model.list(nb_orders)

        # All orders must be picked
        model.constraint(model.count(picking_list) == nb_orders)

        # Create an Hexaly array for the distance matrix in order to access it using the "at" operator
        distances_matrix = model.array(distances_data)

        # Lambda expression to compute the distance to the next order
        distance_to_next_order_lambda = model.lambda_function( 
            lambda i : model.at(distances_matrix, picking_list[i], picking_list[i + 1]))

        # The objective is to minimize the total distance required to pick 
        # all the orders and to go back to the initial position
        objective = model.sum(model.range(0, nb_orders - 1), distance_to_next_order_lambda) \
            + model.at(distances_matrix, picking_list[nb_orders - 1], picking_list[0])

        # Store the index of the initial position in the list.
        # It will be used at the end to write the solution starting from the initial point.
        index_initial_position = model.index(picking_list, 0)

        model.minimize(objective)

        # End of the model declaration
        model.close()

        optimizer.param.time_limit = time_limit

        optimizer.solve()

        if output_file != None :
            with open(output_file, 'w') as f:
                f.write("%i\n" % objective.value)
                for i in range(nb_orders):
                    index = (index_initial_position.get_value() + i) % nb_orders
                    f.write("%i " % picking_list.value[index])


if __name__ == '__main__' :
    if len(sys.argv) < 2:
        print("Usage: python order_picking.py input_file [output_file] [time_limit]")
        sys.exit(1)
    input_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 10
    main(input_file, output_file, time_limit)
